# 02 · Guardian (lever 1)

**What changed (baseline to v1):** red-flag escalation moves out of the model and into code. `nestling/guardian.py` applies red-flag rules written from verified source pages (with negation and general-question handling), flags unsafe requests (doses, medicines, scan interpretation, diagnosis), and lets a Haiku classifier only *propose* flags, which count only when its quote appears verbatim in the user's message. `decide()` and `screen()` make the call; an LLM never decides escalation alone (constitution I). Escalations and refusals are answered by rules, with no model call.

**Hypothesis (plan §9.5 #1):** escalation recall to 100% and fewer missed escalations.

The pipeline wiring is in `nestling/graph.py` (v1 = Guardian in front of the baseline answerer). This notebook only reads saved results.

In [1]:
from evals import compare

table = compare.comparison_table()
BEFORE, AFTER = "baseline", "v1"
rows = [r for r in table if r["version"] in (BEFORE, AFTER)]
print(compare.format_table(rows))

version                 esc             refuse              route               cite              faith              p95 s            mean Rs             p95 Rs
baseline             100.0%            97.1% X            84.6% X              78.1%            89.3% X             8.57 X              1.305              1.502
v1           100.0% (+0.0%)    99.0% (+1.9%) X    79.8% (-4.8%) X      82.2% (+4.1%)    85.7% (-3.6%) X       7.58 (-0.99)     1.007 (-0.298)     1.513 (+0.011)
X = misses the §9.1 pass bar (citation match has no bar of its own); deltas are vs the previous row


## Metrics before and after

Saved results only: `evals/results/*_summary.json` plus the Nebius judge CSVs. Deltas are AFTER minus BEFORE; X marks a missed §9.1 pass bar.

In [2]:
for v in ("baseline", "v1"):
    f = compare.load_summary(v)["failures"]
    print(v, {k: [x["case_id"] for x in items] for k, items in f.items()})

baseline {'missed_escalations': [], 'false_escalations': ['M15'], 'wrong_refusals': ['B11', 'B18', 'E05'], 'errors': []}
v1 {'missed_escalations': [], 'false_escalations': [], 'wrong_refusals': ['M11'], 'errors': []}


## Reading the result

- **Escalation recall stayed at 100% (22/22), so the hypothesis was not a measured gain.** The baseline model had already self-reported every red flag. What changed is the guarantee: in v1 the decision is made by code, so one wrong model call can no longer go unnoticed.
- **Over-escalation fixed:** the baseline's false escalation (M15, herbal kadha for contractions) is gone; false escalations 1 to 0.
- **Refusal calibration improved:** wrong refusals 3 (B11, B18, E05) to 1 (M11); refusal accuracy 97.1% to 99.0%, still short of the 100% bar.
- **Cost and latency fell** (mean 1.305 to 1.007 INR, p95 8.57 s to 7.58 s), consistent with rule-handled cases skipping the model call.
- **Not improved, not targeted by this lever:** routing 84.6% to 79.8% and faithfulness 89.3% to 85.7% (judge). The Guardian does not touch routing or answer text; these moves are model variance on the unchanged answerer and are addressed by later levers.

## Live run (off)

Re-running costs money (see the cost column) and writes a new LangSmith experiment. Saved results above are what this notebook reports.

In [3]:
RUN_LIVE = False

if RUN_LIVE:
    from dotenv import find_dotenv, load_dotenv

    load_dotenv(find_dotenv(usecwd=True))
    from nestling import graph

    graph.run_experiment("v1")
else:
    print("RUN_LIVE = False: showing saved results only")

RUN_LIVE = False: showing saved results only
